# 3. Análisis de los 27 ratones (conducta)
Sube `data_train.zip` (de la notebook 1) y `sessions_used.csv`. Orden: modelos de valor y DMA → estadísticas extra y latencia → modelos de recuperación episódica / HMM / valor+CK.

In [ ]:
!pip -q install numba scipy pandas matplotlib numpy

In [ ]:
%%writefile models.py
"""Trial-level models of lick / withhold behaviour in a change-detection task.

All models share one observation model and the same nuisance terms; they differ ONLY in how
the value difference and the inverse temperature evolve:

    logit P(lick_t) = beta_t * (V_lick - V_withhold) + b + w * sat_t
    sat_t           = cumulative reward before trial t, divided by 100  (satiety / motivation proxy)

bias      no learning: logit = b + w*sat
q         TD learning, fixed alpha, fixed beta                       (= DMA with the gate switched off)
ph        Pearce-Hall style: surprise gates the LEARNING RATE
dma       surprise gates the INVERSE TEMPERATURE through a bAP/HCN conductance g_t
          (the discrete DMA mapping of the reference implementation); alpha, beta_base, lambda, tau_hcn free
dma_const same equations with the constants published for DMA, only (b, w) fitted

Teacher forcing: the observed action drives learning; we score its probability.
Reward = 1 only for a lick on a go trial (as in the task); delta uses the chosen action's value
exactly like the reference implementation.
"""
from __future__ import annotations

import math

import numpy as np

try:  # numba is optional; without it everything still runs, only much slower
    from numba import njit
except Exception:  # pragma: no cover
    def njit(*a, **k):
        if len(a) == 1 and callable(a[0]):
            return a[0]
        return lambda f: f

# Published DMA constants (reference implementation)
ALPHA_C, TAU_C, GMAX, VMAX, LAMBDA_C, BETA_C = 0.15, 45.0, 1.0, 1.0, 0.8, 5.5

MODELS = ["bias", "q", "ph", "dma", "dma_const", "ck", "epi", "hmm", "hmmck", "qck", "dmack"]
MID = {m: i for i, m in enumerate(MODELS)}

# name, bounds, starting points
SPECS = {
    "bias": (["b", "w", "s"], [(-14, 14), (-8, 8), (0, 12)], [[0.0, 0.0, 3.0]]),
    "q": (["alpha", "beta", "b", "w", "s"], [(0.01, 0.9), (0.2, 15), (-14, 14), (-8, 8), (0, 12)],
          [[0.15, 4.0, -4.0, 0.0, 3.0], [0.4, 8.0, -7.0, -1.0, 4.0]]),
    "ph": (["alpha0", "kappa", "eta", "beta", "b", "w", "s"],
           [(0.01, 0.9), (0.0, 5.0), (0.01, 0.9), (0.2, 15), (-14, 14), (-8, 8), (0, 12)],
           [[0.15, 1.0, 0.2, 4.0, -4.0, 0.0, 3.0], [0.3, 0.3, 0.5, 8.0, -7.0, -1.0, 4.0]]),
    "dma": (["alpha", "beta_base", "lam", "tau_hcn", "b", "w", "s"],
            [(0.01, 0.9), (0.2, 15), (0.0, 1.5), (3.0, 300.0), (-14, 14), (-8, 8), (0, 12)],
            [[0.15, 5.5, 0.8, 45.0, -5.0, 0.0, 3.0], [0.3, 8.0, 0.4, 20.0, -7.0, -1.0, 4.0]]),
    "dma_const": (["b", "w", "s"], [(-14, 14), (-8, 8), (0, 12)], [[-5.0, 0.0, 3.0], [-7.0, -1.0, 4.0]]),
    # episodic-retrieval models (adapted from the reference implementation's episodic memory):
    # ck  : recency-weighted retrieval of past choices made under the same cue (a plain choice kernel; control)
    # epi : ck plus retrieval of "problem" engrams (a lick that was not rewarded) under the same cue
    "ck": (["rho", "g1", "b", "w", "s"], [(0.3, 0.999), (-12, 12), (-14, 14), (-8, 8), (0, 12)],
           [[0.9, 1.0, -4.0, 0.0, 3.0], [0.6, 2.0, -3.0, 0.0, 3.0]]),
    # latent engagement: two-state sticky HMM, state-specific bias b_k and cue gain s_k (cf. Ashwood et al. 2022)
    "hmm": (["p11", "p22", "b1", "b2", "s1", "s2", "w"],
            [(0.5, 0.999), (0.5, 0.999), (-14, 14), (-14, 14), (0, 12), (0, 12), (-8, 8)],
            [[0.95, 0.95, -6.0, -2.0, 3.0, 3.0, 0.0], [0.99, 0.9, -8.0, -4.0, 4.0, 2.0, -1.0]]),
    "hmmck": (["p11", "p22", "b1", "b2", "s1", "s2", "w", "rho", "g1"],
              [(0.5, 0.999), (0.5, 0.999), (-14, 14), (-14, 14), (0, 12), (0, 12), (-8, 8), (0.3, 0.999), (-12, 12)],
              [[0.95, 0.95, -6.0, -2.0, 3.0, 3.0, 0.0, 0.85, 1.0], [0.99, 0.9, -8.0, -4.0, 4.0, 2.0, -1.0, 0.6, 2.0]]),
    # value learning PLUS choice retrieval (perseveration): TD or DMA gate on the value term, CK term added to the logit
    "qck": (["alpha", "beta", "rho", "g1", "b", "w", "s"],
            [(0.01, 0.9), (0.2, 15), (0.3, 0.999), (-12, 12), (-14, 14), (-8, 8), (0, 12)],
            [[0.15, 4.0, 0.85, 1.0, -4.0, 0.0, 3.0], [0.4, 8.0, 0.6, 2.0, -7.0, -1.0, 4.0]]),
    "dmack": (["alpha", "beta_base", "lam", "tau_hcn", "rho", "g1", "b", "w", "s"],
              [(0.01, 0.9), (0.2, 15), (0.0, 1.5), (3.0, 300.0), (0.3, 0.999), (-12, 12), (-14, 14), (-8, 8), (0, 12)],
              [[0.15, 5.5, 0.8, 45.0, 0.85, 1.0, -5.0, 0.0, 3.0], [0.3, 8.0, 0.4, 20.0, 0.6, 2.0, -7.0, -1.0, 4.0]]),
    "epi": (["rho", "g1", "g2", "b", "w", "s"], [(0.3, 0.999), (-12, 12), (-12, 12), (-14, 14), (-8, 8), (0, 12)],
            [[0.9, 1.0, -1.0, -4.0, 0.0, 3.0], [0.6, 2.0, 0.0, -3.0, 0.0, 3.0]]),
}


@njit(cache=True)
def _core(mid, P, action, reward, sat, u, rewarded, cue, state, simulate, ll, beta_tr, delta_tr, g_tr):
    n = action.shape[0]
    v0 = 0.33
    V = np.full((2, 2), v0)   # V[state, action]
    g = GMAX
    sm = 0.0
    cum = 0.0
    num_a = np.zeros(2); num_p = np.zeros(2); den = np.zeros(2)   # per-cue episodic traces (models ck, epi)
    for t in range(n):
        s = cum / 100.0 if simulate else sat[t]
        st = state[t]
        V0 = V[st, 0]
        V1 = V[st, 1]
        # ---- policy -------------------------------------------------------------
        if mid == 0:
            beta = 1.0
            z = P[0] + P[1] * s
        elif mid == 1:
            beta = P[1]
            z = beta * (V1 - V0) + P[2] + P[3] * s
        elif mid == 2:
            beta = P[3]
            z = beta * (V1 - V0) + P[4] + P[5] * s
        elif mid == 3:
            beta = P[1] * g / GMAX
            z = beta * (V1 - V0) + P[4] + P[5] * s
        elif mid == 4:
            beta = BETA_C * g / GMAX
            z = beta * (V1 - V0) + P[0] + P[1] * s
        elif mid == 9:
            beta = P[1]
            cc = cue[t]
            ma = num_a[cc] / den[cc] if den[cc] > 0 else 0.0
            z = beta * (V1 - V0) + P[3] * ma + P[4] + P[5] * s
        elif mid == 10:
            beta = P[1] * g / GMAX
            cc = cue[t]
            ma = num_a[cc] / den[cc] if den[cc] > 0 else 0.0
            z = beta * (V1 - V0) + P[5] * ma + P[6] + P[7] * s
        elif mid == 5:
            beta = 1.0
            cc = cue[t]
            ma = num_a[cc] / den[cc] if den[cc] > 0 else 0.0
            z = P[1] * ma + P[2] + P[3] * s
        else:
            beta = 1.0
            cc = cue[t]
            ma = num_a[cc] / den[cc] if den[cc] > 0 else 0.0
            mp = num_p[cc] / den[cc] if den[cc] > 0 else 0.0
            z = P[1] * ma + P[2] * mp + P[3] + P[4] * s
        z += P[P.shape[0] - 1] * cue[t]
        if z > 30.0:
            z = 30.0
        elif z < -30.0:
            z = -30.0
        pl = 1.0 / (1.0 + math.exp(-z))
        if simulate:
            a = 1 if u[t] < pl else 0
            r = 1 if (a == 1 and rewarded[t] == 1) else 0
            action[t] = a
            reward[t] = r
            sat[t] = s
        else:
            a = action[t]
            r = reward[t]
        pa = pl if a == 1 else 1.0 - pl
        if pa < 1e-12:
            pa = 1e-12
        ll[t] = math.log(pa)
        beta_tr[t] = beta
        g_tr[t] = g
        cum += r
        if mid >= 9:
            cc = cue[t]
            rho = P[2] if mid == 9 else P[4]
            num_a[cc] = rho * num_a[cc] + (2.0 * a - 1.0)
            den[cc] = rho * den[cc] + 1.0
        if mid == 0:
            delta_tr[t] = 0.0
            continue
        if mid == 5 or mid == 6:  # store this trial as an engram of its cue and decay the older ones
            cc = cue[t]
            rho = P[0]
            num_a[cc] = rho * num_a[cc] + (2.0 * a - 1.0)
            num_p[cc] = rho * num_p[cc] + (1.0 if (a == 1 and r == 0) else 0.0)
            den[cc] = rho * den[cc] + 1.0
            delta_tr[t] = 0.0
            continue
        # ---- learning -----------------------------------------------------------
        Va = V1 if a == 1 else V0
        delta = r - Va
        delta_tr[t] = delta
        if mid == 1 or mid == 9 or mid == 10:
            a_t = P[0]
        elif mid == 2:
            a_t = P[0] * (1.0 + P[1] * sm)
            if a_t > 0.95:
                a_t = 0.95
            sm = (1.0 - P[2]) * sm + P[2] * abs(delta)
        elif mid == 3:
            a_t = P[0]
        else:
            a_t = ALPHA_C
        Vn = Va + a_t * delta
        if mid == 3 or mid == 4 or mid == 10:  # DMA clips values to [0,1]
            Vn = min(1.0, max(0.0, Vn))
        V[st, a] = Vn
        if mid == 3 or mid == 4 or mid == 10:
            lam = P[2] if (mid == 3 or mid == 10) else LAMBDA_C
            tau = P[3] if (mid == 3 or mid == 10) else TAU_C
            v_ap = VMAX * (1.0 - lam * abs(delta))
            phi = 1.0 / (1.0 + math.exp(-(v_ap - 0.45) / 0.12))
            g = g + (1.0 / tau) * (GMAX * phi - g)
            if g < 0.01:
                g = 0.01
            elif g > GMAX:
                g = GMAX


@njit(cache=True)
def _hmm(P, action, sat, cue, use_ck, ll):
    n = action.shape[0]
    p11 = P[0]; p22 = P[1]
    pi1 = (1.0 - p22) / (2.0 - p11 - p22)
    pr = np.array([pi1, 1.0 - pi1])
    num = np.zeros(2); den = np.zeros(2)
    for t in range(n):
        cc = cue[t]
        ma = 0.0
        if use_ck and den[cc] > 0:
            ma = num[cc] / den[cc]
        pl = 0.0
        pk = np.zeros(2)
        for k in range(2):
            z = P[2 + k] + P[4 + k] * cue[t] + P[6] * sat[t]
            if use_ck:
                z += P[8] * ma
            if z > 30.0: z = 30.0
            elif z < -30.0: z = -30.0
            pk[k] = 1.0 / (1.0 + math.exp(-z))
            pl += pr[k] * pk[k]
        a = action[t]
        pa = pl if a == 1 else 1.0 - pl
        if pa < 1e-12: pa = 1e-12
        ll[t] = math.log(pa)
        po0 = pr[0] * (pk[0] if a == 1 else 1.0 - pk[0])
        po1 = pr[1] * (pk[1] if a == 1 else 1.0 - pk[1])
        sm = po0 + po1
        if sm < 1e-300: sm = 1e-300
        po0 /= sm; po1 /= sm
        pr[0] = po0 * p11 + po1 * (1.0 - p22)
        pr[1] = po0 * (1.0 - p11) + po1 * p22
        if use_ck:
            rho = P[7]
            num[cc] = rho * num[cc] + (2.0 * a - 1.0)
            den[cc] = rho * den[cc] + 1.0


def loglik(model, params, action, reward, sat, cue, state=None, traces=False):
    n = len(action)
    if model in ("hmm", "hmmck"):
        ll = np.empty(n)
        _hmm(np.asarray(params, float), np.array(action, dtype=np.int64), np.asarray(sat, float),
             np.asarray(cue, np.int64), model == "hmmck", ll)
        return ll
    state = np.zeros(n, np.int64) if state is None else np.asarray(state, np.int64)
    ll = np.empty(n); b = np.empty(n); d = np.empty(n); g = np.empty(n)
    _core(MID[model], np.asarray(params, float), np.array(action, dtype=np.int64),
          np.array(reward, dtype=np.int64), np.asarray(sat, float), np.zeros(n),
          np.zeros(n, np.int64), np.asarray(cue, np.int64), state, False, ll, b, d, g)
    return (ll, dict(beta=b, delta=d, g=g)) if traces else ll


def simulate(model, params, rewarded, rng, cue=None, state=None):
    n = len(rewarded)
    cue = rewarded if cue is None else cue
    state = np.zeros(n, np.int64) if state is None else np.asarray(state, np.int64)
    act = np.zeros(n, np.int64); rew = np.zeros(n, np.int64); sat = np.zeros(n)
    ll = np.empty(n); b = np.empty(n); d = np.empty(n); g = np.empty(n)
    _core(MID[model], np.asarray(params, float), act, rew, sat, rng.random(n),
          np.asarray(rewarded, np.int64), np.asarray(cue, np.int64), state, True, ll, b, d, g)
    return act, rew, sat


In [ ]:
%%writefile fit.py
"""Fitting and leave-one-session-out cross-validation (per mouse)."""
from __future__ import annotations

import numpy as np
from scipy.optimize import minimize

import models as M


def nll(params, model, sessions):
    tot = 0.0
    for act, rew, sat, cue, st in sessions:
        tot -= M.loglik(model, params, act, rew, sat, cue, st).sum()
    return tot


def fit(model, sessions, extra_starts=0, rng=None):
    names, bounds, starts = M.SPECS[model]
    starts = [np.array(s, float) for s in starts]
    if extra_starts:
        rng = np.random.default_rng(0) if rng is None else rng
        lo = np.array([b[0] for b in bounds]); hi = np.array([b[1] for b in bounds])
        for _ in range(extra_starts):
            starts.append(lo + rng.random(len(lo)) * (hi - lo))
    best = None
    for s0 in starts:
        r = minimize(nll, s0, args=(model, sessions), method="L-BFGS-B", bounds=bounds)
        if best is None or r.fun < best.fun:
            best = r
    return best.x, best.fun


def loso(mouse_sessions, models_list=None, extra_starts=0):
    """mouse_sessions: list of (act, rew, sat) for ONE mouse. Returns dict model -> array of
    held-out mean log-lik per trial for each session (and the fitted params per fold)."""
    models_list = models_list or M.MODELS
    k = len(mouse_sessions)
    out = {m: np.full(k, np.nan) for m in models_list}
    par = {m: [None] * k for m in models_list}
    if k < 2:
        return out, par
    for i in range(k):
        train = [s for j, s in enumerate(mouse_sessions) if j != i]
        te = mouse_sessions[i]
        for m in models_list:
            p, _ = fit(m, train, extra_starts)
            out[m][i] = M.loglik(m, p, *te).mean()
            par[m][i] = p
    return out, par


def boot_ci(x, n=5000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    rng = np.random.default_rng(seed)
    bs = rng.choice(x, size=(n, len(x))).mean(1)
    return float(x.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


In [ ]:
# Sube data_train.zip
import zipfile
from google.colab import files
up=files.upload()
for k in up:
    if k.endswith('.zip'): zipfile.ZipFile(k).extractall('.')
!ls

In [ ]:
%%writefile run_real.py
"""Real-data analysis: held-out likelihood of q / ph / dma / dma_const on Allen Visual Behavior trials.

Usage:  python run_real.py --data data --out real_results.json
Pre-specified analysis (do not change after looking at the results):
  trials kept   : not aborted, not auto_rewarded
  action        : lick = hit or false_alarm ;  reward = hit ;  cue = go trial
  CV            : leave-one-session-out within mouse (>= 3 sessions per mouse)
  primary tests : (1) dma - q   (2) dma - ph   (3) dma - dma_const   mouse-level mean + bootstrap 95% CI
  secondary     : per-mouse fitted dma on familiar vs novel sessions (paired), identifiability report
"""
import argparse, json, os
import numpy as np
import pandas as pd
import models as M
import fit as F

FIT = ["bias", "q", "ph", "dma", "dma_const"]


def load_session(path):
    t = pd.read_csv(path)
    for c in ["aborted", "auto_rewarded"]:
        if c not in t.columns:
            t[c] = False
    t = t[(~t["aborted"].astype(bool)) & (~t["auto_rewarded"].astype(bool))]
    t = t[t["go"].astype(bool) | t["catch"].astype(bool)]
    lick = (t["hit"].astype(bool) | t["false_alarm"].astype(bool)).astype(int).to_numpy()
    rew = t["hit"].astype(bool).astype(int).to_numpy()
    cue = t["go"].astype(bool).astype(int).to_numpy()
    sat = np.concatenate([[0], np.cumsum(rew)[:-1]]) / 100.0
    return lick, rew, sat, cue, np.zeros(len(lick), int)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="real_results.json")
    ap.add_argument("--min_trials", type=int, default=150)
    ap.add_argument("--min_hit_rate", type=float, default=0.0, help="sensitivity analysis: drop disengaged sessions")
    a = ap.parse_args()
    sess = pd.read_csv(os.path.join(a.data, "sessions.csv"))
    per_mouse = {}
    for _, r in sess.iterrows():
        p = os.path.join(a.data, "trials_%d.csv" % r["behavior_session_id"])
        if not os.path.exists(p):
            continue
        s = load_session(p)
        go = s[3] == 1
        hr = s[1][go].mean() if go.any() else 0.0
        if len(s[0]) >= a.min_trials and hr >= a.min_hit_rate:
            per_mouse.setdefault(r["mouse_id"], []).append((int(r["novel_images"]), s))
    ll = {m: [] for m in FIT}; par_f, par_n = [], []
    for mouse, items in per_mouse.items():
        S = [s for _, s in items]
        if len(S) < 3:
            continue
        out, _ = F.loso(S, FIT)
        for m in FIT:
            ll[m].append(float(np.nanmean(out[m])))
        fam = [s for nv, s in items if nv == 0]; nov = [s for nv, s in items if nv == 1]
        if len(fam) >= 1 and len(nov) >= 1:
            pf, _ = F.fit("dma", fam, extra_starts=2); pn, _ = F.fit("dma", nov, extra_starts=2)
            par_f.append(pf.tolist()); par_n.append(pn.tolist())
    d = lambda x, y: F.boot_ci(np.array(ll[x]) - np.array(ll[y]))
    res = {"n_mice": len(ll["q"]), "mean_ll": {m: float(np.mean(ll[m])) for m in FIT},
           "dma_minus_q": d("dma", "q"), "dma_minus_ph": d("dma", "ph"), "dma_minus_dmaconst": d("dma", "dma_const"),
           "ph_minus_q": d("ph", "q"), "q_minus_bias": d("q", "bias"),
           "per_mouse_ll": ll, "dma_params_familiar": par_f, "dma_params_novel": par_n,
           "dma_param_names": M.SPECS["dma"][0]}
    json.dump(res, open(a.out, "w"), indent=1)
    print(json.dumps({k: v for k, v in res.items() if k not in ("per_mouse_ll", "dma_params_familiar", "dma_params_novel")}, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile analysis_extra.py
"""Extra real-data statistics for the paper: sample description, in-sample gain of the gate, parameter bounds."""
import json, os, glob, sys
import numpy as np, pandas as pd
import models as M, fit as F, run_real as R

D = sys.argv[1]
sess = pd.read_csv(os.path.join(D, "sessions.csv"))
out = {}
for tag, mh in [("primary", 0.0), ("engaged", 0.10)]:
    per = {}
    for _, r in sess.iterrows():
        p = os.path.join(D, "trials_%d.csv" % r["behavior_session_id"])
        if not os.path.exists(p): continue
        s = R.load_session(p); go = s[3] == 1
        hr = s[1][go].mean() if go.any() else 0.0
        if len(s[0]) >= 150 and hr >= mh:
            per.setdefault(r["mouse_id"], []).append(s)
    per = {k: v for k, v in per.items() if len(v) >= 3}
    n_sess = sum(len(v) for v in per.values()); n_tr = sum(len(s[0]) for v in per.values() for s in v)
    hit = np.mean([s[1][s[3] == 1].mean() for v in per.values() for s in v])
    fa = np.nanmean([ (s[0][s[3] == 0].mean() if (s[3] == 0).any() else np.nan) for v in per.values() for s in v])
    rows = []
    for mouse, S in per.items():
        pq, nq = F.fit("q", S, extra_starts=3); pd_, nd = F.fit("dma", S, extra_starts=3); pp, npp = F.fit("ph", S, extra_starts=3)
        names = M.SPECS["dma"][0]; b = dict(zip(names, pd_)); bnds = dict(zip(names, M.SPECS["dma"][1]))
        at_bound = [k for k in ("lam", "tau_hcn") if min(abs(b[k] - bnds[k][0]), abs(b[k] - bnds[k][1])) < 1e-2 * (bnds[k][1] - bnds[k][0])]
        rows.append(dict(mouse=int(mouse), n_sessions=len(S), gain_dma_over_q_nats=float(nq - nd), gain_ph_over_q_nats=float(nq - npp),
                         lam=float(b["lam"]), tau=float(b["tau_hcn"]), beta_base=float(b["beta_base"]), alpha=float(b["alpha"]), at_bound=at_bound))
    out[tag] = dict(n_mice=len(per), n_sessions=n_sess, n_trials=int(n_tr), mean_hit=float(hit), mean_fa=float(fa), mice=rows)
    print(tag, out[tag]["n_mice"], "mice", n_sess, "sessions", n_tr, "trials; hit %.2f FA %.2f" % (hit, fa))
    for r in rows: print("  ", r)
json.dump(out, open("real_extra.json", "w"), indent=1)


In [ ]:
%%writefile latency_test.py
"""Does the DMA gate g_t (published constants) predict response latency on lick trials?
Prediction of a gate that scales inverse temperature: after a surprise g falls, responding becomes less
deterministic/slower, so within-session log-latency should relate to g_t (expected sign: negative).
Pre-specified: slope of z(log latency) on z(g_t), controlling z(sat) and z(trial index), within session;
mouse-level mean slope + bootstrap CI; null = circular shift of g within session (keeps autocorrelation).
Usage: python latency_test.py DATA OUT.json"""
import sys, os, json
import numpy as np, pandas as pd
import models as M, fit as F, run_real as R

D, OUT = sys.argv[1], sys.argv[2]
rng = np.random.default_rng(0)
sess = pd.read_csv(os.path.join(D, "sessions.csv"))
P = np.array([-5.0, 0.0, 3.0])  # dma_const params only matter for the gate trace, which depends on observed actions/rewards only


def z(x):
    s = x.std()
    return (x - x.mean()) / s if s > 0 else x * 0


def slope(y, X):
    X = np.column_stack([np.ones(len(y)), X])
    return np.linalg.lstsq(X, y, rcond=None)[0][1]


per = {}
for _, r in sess.iterrows():
    p = os.path.join(D, "trials_%d.csv" % r["behavior_session_id"])
    if not os.path.exists(p): continue
    t = pd.read_csv(p)
    for c in ["aborted", "auto_rewarded"]:
        if c not in t.columns: t[c] = False
    t = t[(~t["aborted"].astype(bool)) & (~t["auto_rewarded"].astype(bool))]
    t = t[t["go"].astype(bool) | t["catch"].astype(bool)].reset_index(drop=True)
    lick, rew, sat, cue, st = R.load_session(p)
    if len(lick) < 150: continue
    _, tr = M.loglik("dma_const", P, lick, rew, sat, cue, traces=True)
    lat = pd.to_numeric(t["response_latency"], errors="coerce").to_numpy()
    ok = (lick == 1) & np.isfinite(lat) & (lat > 0.1) & (lat < 1.0)
    if ok.sum() < 30: continue
    g = tr["g"]; idx = np.arange(len(lick))
    per.setdefault(r["mouse_id"], []).append((np.log(lat), g, sat, idx, ok))

res = []
for m, L in per.items():
    if len(L) < 3: continue
    obs, nul = [], []
    for ll, g, sat, idx, ok in L:
        y = z(ll[ok]); X = np.column_stack([z(sat[ok]), z(idx[ok])])
        obs.append(slope(y, np.column_stack([z(g[ok]), X])))
        k = int(rng.integers(20, len(g) - 20)); gs = np.roll(g, k)
        nul.append(slope(y, np.column_stack([z(gs[ok]), X])))
    res.append((m, float(np.mean(obs)), float(np.mean(nul)), len(L)))
obs = np.array([r[1] for r in res]); nul = np.array([r[2] for r in res])
# null distribution of the mouse-mean slope: many circular-shift replicates
reps = []
for _ in range(500):
    s = []
    for m, L in per.items():
        if len(L) < 3: continue
        v = []
        for ll, g, sat, idx, ok in L:
            y = z(ll[ok]); X = np.column_stack([z(sat[ok]), z(idx[ok])])
            k = int(rng.integers(20, len(g) - 20))
            v.append(slope(y, np.column_stack([z(np.roll(g, k)[ok]), X])))
        s.append(np.mean(v))
    reps.append(np.mean(s))
reps = np.array(reps)
mean = obs.mean(); ci = F.boot_ci(obs)
p = float((np.abs(reps) >= abs(mean)).mean())
out = dict(n_mice=len(res), mean_slope=float(mean), ci=list(map(float, ci)), null_sd=float(reps.std()), p_shift=p,
           frac_negative=float((obs < 0).mean()), per_mouse=obs.tolist())
json.dump(out, open(OUT, "w"), indent=1); print({k: v for k, v in out.items() if k != "per_mouse"})


In [ ]:
%%writefile run_episodic.py
"""Held-out comparison of the episodic-retrieval models (ck, epi) against TD and bias, same protocol as run_real.py.
Usage: python run_episodic.py DATA OUT.json [min_hit_rate]"""
import sys, os, json
import numpy as np, pandas as pd
import fit as F, run_real as R

D, OUT = sys.argv[1], sys.argv[2]; mh = float(sys.argv[3]) if len(sys.argv) > 3 else 0.0
sess = pd.read_csv(os.path.join(D, "sessions.csv")); per = {}
for _, r in sess.iterrows():
    p = os.path.join(D, "trials_%d.csv" % r["behavior_session_id"])
    if not os.path.exists(p): continue
    s = R.load_session(p); go = s[3] == 1
    hr = s[1][go].mean() if go.any() else 0.0
    if len(s[0]) >= 150 and hr >= mh: per.setdefault(r["mouse_id"], []).append(s)
FITM = ["q", "bias", "ck", "epi"]; ll = {m: [] for m in FITM}; par = []
for mouse, S in per.items():
    if len(S) < 3: continue
    out, _ = F.loso(S, FITM)
    for m in FITM: ll[m].append(float(np.nanmean(out[m])))
    pe, _ = F.fit("epi", S, extra_starts=2); par.append(pe.tolist())
d = lambda a, b: F.boot_ci(np.array(ll[a]) - np.array(ll[b]))
res = dict(n_mice=len(ll["q"]), mean_ll={m: float(np.mean(ll[m])) for m in FITM},
           ck_minus_q=d("ck", "q"), epi_minus_q=d("epi", "q"), epi_minus_ck=d("epi", "ck"), ck_minus_bias=d("ck", "bias"),
           epi_minus_bias=d("epi", "bias"), per_mouse_ll=ll, epi_params=par)
json.dump(res, open(OUT, "w"), indent=1)
print({k: ([round(x * 1e4, 1) for x in v] if k.endswith(("_q", "_ck", "_bias")) else v) for k, v in res.items() if k not in ("per_mouse_ll", "epi_params")})


In [ ]:
%%writefile run_hmm.py
"""Held-out comparison of latent-engagement (2-state HMM) models against the other models, same protocol as run_real.py.
Mouse order is identical to run_episodic.py, so per-mouse values can be paired with epi30_*.json.
Usage: python run_hmm.py DATA OUT.json [min_hit_rate]"""
import sys, os, json
import numpy as np, pandas as pd
import fit as F, run_real as R

D, OUT = sys.argv[1], sys.argv[2]; mh = float(sys.argv[3]) if len(sys.argv) > 3 else 0.0
ref = sys.argv[4] if len(sys.argv) > 4 else None
sess = pd.read_csv(os.path.join(D, "sessions.csv")); per = {}
for _, r in sess.iterrows():
    p = os.path.join(D, "trials_%d.csv" % r["behavior_session_id"])
    if not os.path.exists(p): continue
    s = R.load_session(p); go = s[3] == 1
    hr = s[1][go].mean() if go.any() else 0.0
    if len(s[0]) >= 150 and hr >= mh: per.setdefault(r["mouse_id"], []).append(s)
FITM = ["hmm", "hmmck"]; ll = {m: [] for m in FITM}; par = []
for mouse, S in per.items():
    if len(S) < 3: continue
    out, _ = F.loso(S, FITM, extra_starts=1)
    for m in FITM: ll[m].append(float(np.nanmean(out[m])))
    pe, _ = F.fit("hmmck", S, extra_starts=2); par.append(pe.tolist())
res = dict(n_mice=len(ll["hmm"]), mean_ll={m: float(np.mean(ll[m])) for m in FITM}, per_mouse_ll=ll, hmmck_params=par)
if ref:
    r = json.load(open(ref)); L = r["per_mouse_ll"]; d = lambda a, b: F.boot_ci(np.array(a) - np.array(b))
    res.update(hmm_minus_q=d(ll["hmm"], L["q"]), hmm_minus_ck=d(ll["hmm"], L["ck"]), hmmck_minus_hmm=d(ll["hmmck"], ll["hmm"]),
               hmmck_minus_ck=d(ll["hmmck"], L["ck"]), hmmck_minus_epi=d(ll["hmmck"], L["epi"]))
json.dump(res, open(OUT, "w"), indent=1)
print({k: ([round(x * 1e4, 1) for x in v] if isinstance(v, list) and k.endswith(("_q", "_ck", "_hmm", "_epi")) else v) for k, v in res.items() if k not in ("per_mouse_ll", "hmmck_params")})


In [ ]:
%%writefile run_value_ck.py
"""Value learning WITH choice retrieval: does the DMA gate (or any value learning) add anything once perseveration is modeled?
Same protocol as run_real.py; mouse order identical to run_episodic.py / run_hmm.py (paired with epi30_*.json).
Usage: python run_value_ck.py DATA OUT.json MIN_HIT_RATE REF_EPI_JSON"""
import sys, os, json
import numpy as np, pandas as pd
import fit as F, run_real as R

D, OUT, mh, ref = sys.argv[1], sys.argv[2], float(sys.argv[3]), json.load(open(sys.argv[4]))
sess = pd.read_csv(os.path.join(D, "sessions.csv")); per = {}
for _, r in sess.iterrows():
    p = os.path.join(D, "trials_%d.csv" % r["behavior_session_id"])
    if not os.path.exists(p): continue
    s = R.load_session(p); go = s[3] == 1
    hr = s[1][go].mean() if go.any() else 0.0
    if len(s[0]) >= 150 and hr >= mh: per.setdefault(r["mouse_id"], []).append(s)
FITM = ["qck", "dmack"]; ll = {m: [] for m in FITM}; par = []
for mouse, S in per.items():
    if len(S) < 3: continue
    out, _ = F.loso(S, FITM, extra_starts=1)
    for m in FITM: ll[m].append(float(np.nanmean(out[m])))
    pe, _ = F.fit("dmack", S, extra_starts=2); par.append(pe.tolist())
L = ref["per_mouse_ll"]; d = lambda a, b: F.boot_ci(np.array(a) - np.array(b))
res = dict(n_mice=len(ll["qck"]), mean_ll={m: float(np.mean(ll[m])) for m in FITM}, per_mouse_ll=ll, dmack_params=par,
           dmack_minus_qck=d(ll["dmack"], ll["qck"]), qck_minus_ck=d(ll["qck"], L["ck"]), dmack_minus_ck=d(ll["dmack"], L["ck"]),
           qck_minus_q=d(ll["qck"], L["q"]), dmack_minus_epi=d(ll["dmack"], L["epi"]))
json.dump(res, open(OUT, "w"), indent=1)
print({k: [round(x * 1e4, 1) for x in v] for k, v in res.items() if isinstance(v, (list, tuple)) and k.startswith(("dmack_m", "qck_m"))})


## Ejecución

In [ ]:
# Análisis primario (preespecificado) y de sensibilidad (solo sesiones comprometidas)
!python run_real.py --data data_train --out real30_primary.json
!python run_real.py --data data_train --out real30_engaged.json --min_hit_rate 0.10

In [ ]:
!python analysis_extra.py data_train
!python latency_test.py data_train latency30.json

In [ ]:
!python run_episodic.py data_train epi30_primary.json
!python run_episodic.py data_train epi30_engaged.json 0.10

In [ ]:
!python run_hmm.py data_train hmm30_primary.json 0.0 epi30_primary.json
!python run_hmm.py data_train hmm30_engaged.json 0.10 epi30_engaged.json

In [ ]:
!python run_value_ck.py data_train vck30_primary.json 0.0 epi30_primary.json
!python run_value_ck.py data_train vck30_engaged.json 0.10 epi30_engaged.json

In [ ]:
!zip -q real_results.zip *.json
from google.colab import files
files.download('real_results.zip')